# DataKern — Session 5 Interactive Masterclass
## Spark Internals, Advanced Transformations, Parquet & Delta Lake

**Session 5 · Data Engineering · DataKern**  
*Welcome to the hands-on interactive laboratory. Open your Spark UI at [http://localhost:4040](http://localhost:4040) while running this notebook.*

---
### Masterclass Modules:
1. **Module 1: The Shuffle & Data Skew** (Narrow vs Wide, Exchange nodes, Stage metrics, Skew in São Paulo)
2. **Module 2: Join Strategies, Catalyst & Caching** (Broadcast joins, `.explain()` plan reading, Caching hygiene)
3. **Module 3: Production Schemas, Nulls & UDFs** (Explicit `StructType`, business nulls vs drops, UDF IPC penalty)
4. **Module 4: Window Functions Introduction** (`groupBy` vs `Window`, cumulative totals, ranking)
5. **Module 5: Parquet Anatomy & Partitioning** (Columnar format, disk savings, column pruning, directory partitioning)
6. **Module 6: Delta Lake & Medallion Architecture** (ACID log, Time Travel, History audit, Bronze ➔ Silver ➔ Gold)



### ⚙️ Environment Setup & SparkSession with Delta Lake
Let's initialize our SparkSession with Delta Lake extensions and set `SPARK_LOCAL_IP='127.0.0.1'`.


In [ ]:
import os
import sys
import time
import json
import shutil

# Bind to localhost to prevent macOS network binding issues
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
)
from pyspark.sql.window import Window

DATA_DIR = os.path.join(os.getcwd(), "data")
OUTPUT_DIR = os.path.join(os.getcwd(), "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Build SparkSession with Delta Lake extensions
builder = SparkSession.builder \
    .appName("DataKern_Session5_Masterclass") \
    .master("local[*]") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.ui.port", "4040") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("✅ SparkSession successfully initialized with Delta Lake!")
print("🌐 SPARK UI ACTIVE AT: http://localhost:4040")
print(f"   Default Parallelism (CPU Cores): {spark.sparkContext.defaultParallelism}")
print(f"   💡 Note: local[*] mode uses your laptop's available CPU cores ({spark.sparkContext.defaultParallelism} detected).")
print("      Task counts and partition numbers in your Spark UI will reflect this hardware configuration.")



---
## 📦 Module 1: The Shuffle and Data Skew

### Core Mental Model:
- **Narrow Transformation** (`filter`, `select`, `withColumn`): Each worker processes its own local memory partition. Zero network movement of raw data rows.

- **DataFrame Actions & Aggregations** (`count()`, `sum()`): In Catalyst, `df.count()` runs a 2-phase aggregation (local partial count, followed by a tiny shuffle exchange of partition subtotals to 1 final task to compute the grand total).
- **Wide Transformation** (`groupBy`, `join`, `distinct`): Rows with the same key must be sent to the same worker across the cluster network (`Exchange`). Splits the Job into multiple Stages and shuffles megabytes of actual data records!

- **Data Skew**: When keys are unevenly distributed in the real world (e.g. São Paulo having 41.7% of all Brazilian customers), one worker gets overloaded while others sit idle!



In [ ]:
# 1. Narrow Transformation: Filter orders
df_orders = spark.read.csv(os.path.join(DATA_DIR, "olist_orders_dataset.csv"), header=True, inferSchema=True)

print("Running Narrow Transformation (filter)...")
t0 = time.time()
delivered_orders = df_orders.filter(F.col("order_status") == "delivered")
cnt = delivered_orders.count()
print(f"✅ Filter completed in {time.time() - t0:.3f}s. Total delivered: {cnt:,}")
print("👀 Check Spark UI (http://localhost:4040) -> Jobs Tab: The filter itself moved 0 raw rows. Notice how df.count() uses a 2-stage partial aggregation with only a few bytes of subtotal shuffle!")



In [ ]:
# 2. Wide Transformation: GroupBy customer_state
df_customers = spark.read.csv(os.path.join(DATA_DIR, "olist_customers_dataset.csv"), header=True, inferSchema=True)

print("Running Wide Transformation (groupBy customer_state)...")
t0 = time.time()
state_summary = df_customers.groupBy("customer_state").count().orderBy(F.desc("count"))
state_summary.show(5)
print(f"✅ GroupBy completed in {time.time() - t0:.3f}s.")
print("👀 Check Spark UI -> Stages Tab: Notice 2 Stages separated by an 'Exchange' with Shuffle Read and Write!")



In [ ]:
# 3. Data Skew Demonstration: São Paulo Concentration
total_cust = df_customers.count()
sp_cust = df_customers.filter(F.col("customer_state") == "SP").count()
rr_cust = df_customers.filter(F.col("customer_state") == "RR").count()

print("📊 REAL-WORLD SKEW IN BRAZILIAN DATA:")
print(f"   Total Customers in Brazil : {total_cust:,}")
print(f"   Customers in SP alone     : {sp_cust:,} ({sp_cust / total_cust * 100:.1f}% of entire country!)")
print(f"   Customers in RR (Roraima) : {rr_cust:,}")
print("\n💡 IN PRODUCTION: If you partition by state, the worker processing 'SP' is a straggler task.")
print("   In Spark UI Stages -> Summary Metrics, look for Max Task Duration >> 3 * Median Task Duration.")



---
## ⚡ Module 2: Join Strategies, the Catalyst Optimizer & Caching

### Core Concepts:
1. **SortMergeJoin** (Default): Shuffles and sorts *both* tables across the network.
2. **BroadcastHashJoin** (`F.broadcast`): Driver broadcasts small table (< 100MB) to all executors once. The large fact table streams through with **zero shuffle**!
3. **Catalyst Optimizer**: Pushes filters (`Predicate Pushdown`) and drops unused columns (`Column Pruning`) down to the storage scan.
4. **Caching**: Use `.cache()` *only* for branching dataflows. Always pair with `.unpersist()`.



In [ ]:
df_items = spark.read.csv(os.path.join(DATA_DIR, "olist_order_items_dataset.csv"), header=True, inferSchema=True)
df_products = spark.read.csv(os.path.join(DATA_DIR, "olist_products_dataset.csv"), header=True, inferSchema=True)

# Join with Broadcast Hint
t0 = time.time()
df_joined_broadcast = df_items.join(
    F.broadcast(df_products),
    on="product_id",
    how="inner"
)
joined_count = df_joined_broadcast.count()
print(f"✅ Broadcast Join completed in {time.time() - t0:.3f}s. Total rows: {joined_count:,}")
print("👀 Check Spark UI -> SQL / DataFrame tab: Notice BroadcastHashJoin replaces SortMergeJoin with ZERO shuffle on items!")



In [ ]:
# Inspecting Catalyst Optimizer Plan
df_query = df_items \
    .filter(F.col("price") > 150.0) \
    .select("order_id", "product_id", "price")

print("🔍 PHYSICAL EXECUTION PLAN (Catalyst Optimizer):")
df_query.explain(mode="formatted")
print("\n💡 Look at the Scan step at the bottom:")
print("   - PushedFilters: [IsNotNull(price), GreaterThan(price, 150.0)] -> Predicate Pushdown!")
print("   - ReadSchema: struct<order_id:string,product_id:string,price:double> -> Column Pruning!")



In [ ]:
# Caching Hygiene: Storage Tab and unpersist()
df_branch = df_items.filter(F.col("freight_value") > 30.0).cache()

# Materialize cache
print(f"Action 1 (materialize cache): {df_branch.count():,} rows")
print(f"Action 2 (read from RAM)   : {df_branch.agg(F.avg('price')).collect()[0][0]:.2f} avg price")

print("\n👀 Check Spark UI -> 'Storage' tab: Notice the cached RDD with 100% fraction cached in RAM!")

# Memory Hygiene: Free memory
df_branch.unpersist()
print("✅ df_branch.unpersist() called. Storage tab cleared.")



---
## 🛡️ Module 3: Production Schemas, Nulls & UDF Performance

### Production Hygiene:
1. **Schema Enforcement**: Never rely on `inferSchema=True`. Define explicit `StructType` schemas to avoid schema drift and double file reads.
2. **Nulls as Design Decisions**: An order with `order_status = 'shipped'` has a NULL delivery date because it is **on the truck**! Blindly calling `.dropna()` loses valid in-transit sales.
3. **UDF Bottleneck**: Python UDFs suffer severe IPC socket serialization penalties and blind the Catalyst optimizer. Always use native Spark SQL functions (`F.when`, `F.coalesce`).



In [ ]:
# Explicit Schema Definition
orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_status", StringType(), False),
    StructField("order_purchase_timestamp", StringType(), True),
    StructField("order_approved_at", StringType(), True),
    StructField("order_delivered_carrier_date", StringType(), True),
    StructField("order_delivered_customer_date", StringType(), True),
    StructField("order_estimated_delivery_date", StringType(), True)
])

df_orders_explicit = spark.read.csv(
    os.path.join(DATA_DIR, "olist_orders_dataset.csv"),
    header=True,
    schema=orders_schema,
    mode="DROPMALFORMED"
)
print("✅ Explicit schema applied successfully with zero type guessing!")
df_orders_explicit.printSchema()



In [ ]:
# Null Handling: Business Design Decision
print("Checking status of orders with NULL delivery dates:")
df_orders_explicit.filter(F.col("order_delivered_customer_date").isNull()) \
    .groupBy("order_status") \
    .count() \
    .show()

print("💡 Shipped & processing orders have NULL delivery dates because they are IN-TRANSIT!")
print("   Dropping nulls would delete active pipeline deliveries.")

# Audited Null Cleaning:
total_in = df_orders_explicit.count()
# Drop only invalid primary keys
df_valid = df_orders_explicit.na.drop(subset=["order_id", "customer_id"])
print(f"📊 Audit: Total Ingested={total_in:,} | Corrupt Dropped={total_in - df_valid.count():,} | Preserved={df_valid.count():,}")



In [ ]:
# Benchmarking: Python UDF vs Native Spark Function
def python_tier(price):
    if price is None: return "Unknown"
    elif price < 50.0: return "Budget"
    elif price < 150.0: return "Standard"
    else: return "Premium"

tier_udf = F.udf(python_tier, StringType())

# Approach 1: Python UDF
t0 = time.time()
df_items.withColumn("tier", tier_udf(F.col("price"))).groupBy("tier").count().collect()
time_udf = time.time() - t0

# Approach 2: Native Spark Function
t0 = time.time()
df_items.withColumn(
    "tier",
    F.when(F.col("price").isNull(), "Unknown")
     .when(F.col("price") < 50.0, "Budget")
     .when(F.col("price") < 150.0, "Standard")
     .otherwise("Premium")
).groupBy("tier").count().collect()
time_native = time.time() - t0

print(f"⏱️ Python UDF Time  : {time_udf:.3f}s (Suffers JVM-Python IPC socket serialization)")
print(f"⏱️ Native Spark Time: {time_native:.3f}s (Runs 100% inside compiled JVM)")
print(f"🚀 Speedup: Native Spark is {time_udf / time_native:.1f}x faster!")



---
## 🪟 Module 4: Window Functions Introduction

### Core Mental Model:
- `groupBy`: Summarizes and collapses multiple rows into 1 summary row per group.
- `Window`: Calculates aggregations or rankings across a group **while preserving every original row**.



In [ ]:
# Prepare base dataset
df_base = df_items.select("order_id", "product_id", "price") \
    .join(df_orders.filter(F.col("order_status") == "delivered").select("order_id", "order_purchase_timestamp"), on="order_id") \
    .join(F.broadcast(df_products.select("product_id", "product_category_name")), on="product_id") \
    .filter(F.col("product_category_name").isNotNull())

print(f"Base dataset prepared: {df_base.count():,} items.")



In [ ]:
# Pattern 1: Cumulative Running Category Revenue Over Time
w_running = Window \
    .partitionBy("product_category_name") \
    .orderBy("order_purchase_timestamp") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_running = df_base.withColumn(
    "cumulative_revenue",
    F.round(F.sum("price").over(w_running), 2)
)

print("Sample Running Total for 'telefonia':")
df_running.filter(F.col("product_category_name") == "telefonia") \
    .select("product_category_name", "order_purchase_timestamp", "price", "cumulative_revenue") \
    .show(5, truncate=False)



In [ ]:
# Pattern 2: Top 3 Most Expensive Items per Category (Ranking)
w_rank = Window.partitionBy("product_category_name").orderBy(F.desc("price"))

df_top3 = df_base.withColumn("rank", F.row_number().over(w_rank)) \
    .filter(F.col("rank") <= 3)

print("Top 3 Most Expensive Items in 'audio':")
df_top3.filter(F.col("product_category_name") == "audio") \
    .select("product_category_name", "rank", "price", "order_id") \
    .show(3, truncate=False)



---
## 🗄️ Module 5: Parquet Anatomy & Directory Partitioning

### Why Big Data Abandoned CSV:
1. **Columnar Storage**: Data for each column stored contiguously. Queries reading 2 columns read only those bytes (`Column Pruning`).
2. **Snappy Compression**: Achieves ~70% disk space reduction compared to raw CSV.
3. **Partitioning**: `.partitionBy("order_status")` creates `order_status=delivered/` folder hierarchies. Spark skips reading unneeded folders (`Partition Pruning`).



In [ ]:
parquet_path = os.path.join(OUTPUT_DIR, "orders_parquet")
csv_path = os.path.join(DATA_DIR, "olist_orders_dataset.csv")

# Write Parquet
df_orders.write.mode("overwrite").parquet(parquet_path)

csv_size = os.path.getsize(csv_path) / (1024 * 1024)
parquet_size = sum(os.path.getsize(os.path.join(parquet_path, f)) for f in os.listdir(parquet_path) if f.endswith(".parquet")) / (1024 * 1024)

print(f"📊 DISK FOOTPRINT COMPARISON:")
print(f"   CSV File Size     : {csv_size:.2f} MB")
print(f"   Parquet Directory : {parquet_size:.2f} MB")
print(f"   Storage Reduction : {(1 - parquet_size / csv_size) * 100:.1f}% space saved!")



In [ ]:
# Directory Partitioning by order_status
partitioned_path = os.path.join(OUTPUT_DIR, "orders_partitioned")
df_orders.write.partitionBy("order_status").mode("overwrite").parquet(partitioned_path)

print(f"📁 GENERATED DIRECTORIES ON DISK ({partitioned_path}):")
for d in sorted(os.listdir(partitioned_path)):
    if not d.startswith((".", "_")):
        print(f"   ├── {d}/")

print("\n🔍 PARTITION PRUNING EXECUTION PLAN:")
spark.read.parquet(partitioned_path).filter(F.col("order_status") == "delivered").explain(mode="formatted")
print("💡 Look at PartitionFilters in the scan: Spark completely ignores all other folders!")



---
## 🌊 Module 6: Delta Lake & The Medallion Architecture

### The Formula:
$$\text{Delta Lake} = \text{Parquet Files} + \text{ACID Transaction Log } (\text{_delta_log/})$$

- **ACID Transactions**: Atomic commits. Crashed writes never corrupt data.
- **Time Travel**: Query previous states with `.option("versionAsOf", 0)`.
- **Audit History**: Inspect operations using `DeltaTable.history()`.
- **Medallion Architecture**: Bronze (raw ingest) ➔ Silver (cleaned, joined) ➔ Gold (curated business summary).



In [ ]:
delta_silver = os.path.join(OUTPUT_DIR, "delta_silver_orders")

# Write Version 0 (All orders)
df_orders.select("order_id", "customer_id", "order_status") \
    .write.format("delta").mode("overwrite").save(delta_silver)

print("✅ Version 0 written as Delta Lake.")

# Inspect the Transaction Log JSON
log_file = os.path.join(delta_silver, "_delta_log", "00000000000000000000.json")
print(f"\n📄 INSPECTING TRANSACTION LOG COMMIT ({os.path.basename(log_file)}):")
with open(log_file, "r") as f:
    for i, line in enumerate(f):
        entry = json.loads(line)
        key = list(entry.keys())[0]
        if key in ["commitInfo", "metaData", "protocol"]:
            print(f"   {key:<12} ➔ {entry[key]}")



In [ ]:
# Write Version 1 (Overwriting with delivered orders only)
df_orders.filter(F.col("order_status") == "delivered") \
    .select("order_id", "customer_id", "order_status") \
    .write.format("delta").mode("overwrite").save(delta_silver)

print("✅ Version 1 overwritten.")

# TIME TRAVEL: Query Version 1 vs Version 0!
cnt_v1 = spark.read.format("delta").load(delta_silver).count()
cnt_v0 = spark.read.format("delta").option("versionAsOf", 0).load(delta_silver).count()

print(f"📊 TIME TRAVEL VERIFICATION:")
print(f"   Current Data (Version 1)   : {cnt_v1:,} rows (delivered only)")
print(f"   Historical Data (Version 0): {cnt_v0:,} rows (all orders restored!)")



In [ ]:
# Audit Table History
delta_table = DeltaTable.forPath(spark, delta_silver)
print("📜 DELTA TABLE AUDIT HISTORY:")
delta_table.history().select("version", "timestamp", "operation", "operationMetrics.numOutputRows").show(truncate=False)



In [ ]:
# The Medallion Architecture: Creating Gold Finance Delta Table
delta_gold = os.path.join(OUTPUT_DIR, "delta_gold_finance")

df_silver_orders = spark.read.format("delta").load(delta_silver)

df_gold = df_items \
    .join(df_silver_orders, on="order_id", how="inner") \
    .join(F.broadcast(df_products), on="product_id", how="inner") \
    .filter(F.col("product_category_name").isNotNull()) \
    .groupBy("product_category_name") \
    .agg(
        F.count("order_id").alias("items_sold"),
        F.round(F.sum("price"), 2).alias("total_revenue"),
        F.round(F.avg("price"), 2).alias("avg_price")
    )

w_gold = Window.orderBy(F.desc("total_revenue"))
df_gold = df_gold.withColumn("revenue_rank", F.row_number().over(w_gold))

df_gold.write.format("delta").mode("overwrite").save(delta_gold)

print("🥇 GOLD FINANCE DELTA TABLE (Top 5 Categories):")
spark.read.format("delta").load(delta_gold).show(5, truncate=False)



---
## 🎓 Masterclass Summary & Next Steps

Congratulations! You have completed the **Session 5 Masterclass**:
- ✅ You observed **Shuffle Boundaries** and diagnosed **Data Skew** in the Spark UI.
- ✅ You converted SortMergeJoins to **Broadcast Joins** and validated **Predicate Pushdown** with `.explain()`.
- ✅ You enforced **Production Schemas**, audited **Nulls**, and avoided the **Python UDF serialization trap**.
- ✅ You mastered **Window Functions** without collapsing row granularity.
- ✅ You benchmarked **Parquet Column Pruning** and verified **Partition Pruning** directories.
- ✅ You built ACID Lakehouse tables with **Delta Lake**, executed **Time Travel**, and implemented the **Medallion Architecture**.

### Ready for Assignment 5:
You now have the full toolkit needed to conquer **Assignment 5**!

*Don't focus on getting the answer. Focus on learning how to engineer the solution.*  
— **Uma Kiran | DataKern | [www.data-kern.com](https://www.data-kern.com)**

